# Bruise segmentation -- fixed-test evaluation + benchmark (A100)

Evaluates the **5 core models** on the 185-image `fixed_consensus_test` set,
everything scored on one common $640\times640$ grid, plus a raw-forward speed
benchmark and subject-level (cluster) bootstrap CIs.

**Requires** (already on Drive, no new upload):
```
MyDrive/bruise_segmentation_gpu/bruise_colab_gpu_full.zip
```

### Deliberately excluded
* **No ITA / skin-tone fairness tables.** At 28 test subjects the max-min
  subgroup gap has a cluster-bootstrap 95% CI of roughly [0.04, 0.89] -- too
  wide to report responsibly.
* **No native-camera-resolution scoring.** Everything is scored at 640, so no
  cross-resolution comparison arises.
* **No threshold/temperature calibration path for YOLO.** YOLO is evaluated
  through Ultralytics' native inference only. Feeding YOLO an
  ImageNet-normalised tensor (what the custom raw-logit path did) costs
  ~0.23 mean Dice and doubles the miss rate; that path is not reproduced here.

**Runtime:** Runtime -> Change runtime type -> A100 GPU, before running cell 1.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/bruise_segmentation_gpu'
ZIP_PATH  = f'{DRIVE_DIR}/bruise_colab_gpu_full.zip'

import os
assert os.path.exists(ZIP_PATH), (
    f'{ZIP_PATH} not found -- upload bruise_colab_gpu_full.zip into {DRIVE_DIR}/ first.')
print('Found package:', ZIP_PATH, f'({os.path.getsize(ZIP_PATH)/1e6:.0f} MB)')

## 2. Confirm GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'No CUDA GPU -- Runtime > Change runtime type > A100 GPU, then re-run.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))

## 3. Stage package to local disk
Local disk, not the Drive FUSE mount: reading images over the network would
add unpredictable I/O into a run that also produces timings.

In [ ]:
import shutil, time
LOCAL_DIR = '/content/bruise_gpu'

t0 = time.time()
shutil.copy(ZIP_PATH, '/content/pkg.zip')
print(f'Copied zip in {time.time()-t0:.0f}s')

!rm -rf {LOCAL_DIR}
!mkdir -p {LOCAL_DIR}
t0 = time.time()
!unzip -q /content/pkg.zip -d {LOCAL_DIR}
print(f'Unzipped in {time.time()-t0:.0f}s')

## 4. Install dependencies
torch/CUDA are preinstalled on Colab -- do not reinstall torch.

In [ ]:
%cd {LOCAL_DIR}
!pip install -q transformers ultralytics albumentations opencv-python-headless

## 5. Imports

**`cv2` is imported before `ultralytics` on purpose.** With the reverse order,
`cv2.imread(path, IMREAD_GRAYSCALE)` can return `(H,W,1)` instead of `(H,W)`,
which silently breaks mask handling. Masks are also squeezed defensively below,
so this is belt-and-braces rather than the only guard.

In [ ]:
import cv2                      # MUST precede ultralytics -- see markdown above
import io, json, sys
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from ultralytics import YOLO      # imported after cv2

sys.path.insert(0, LOCAL_DIR)
from pipeline.io_utils import load_yaml
from pipeline.data import BruiseDataset, get_augmentation, load_fixed_test
from pipeline.models import load_segformer_model, count_params
from pipeline.mask_utils import read_gt_mask
from pipeline.metrics import compute_image_row

paths = load_yaml('configs/paths.yaml')
cfg   = load_yaml('configs/common_train.yaml')
IMG_H, IMG_W = cfg['img_h'], cfg['img_w']
assert (IMG_H, IMG_W) == (640, 640), (IMG_H, IMG_W)

test_df = load_fixed_test(paths['fixed_test_manifest'])
print('test images:', len(test_df), '| grid:', IMG_H, 'x', IMG_W)

## 6. Subject IDs (embedded)

Subject IDs come from `ita_labels/wl_test_per_image_ita.csv`, which is **not**
in the zip, so the mapping is embedded here rather than forcing an 868 MB
re-upload. It is *not* regex-derived from the stems: the test set contains both
`TAM009` and `TAM0009` as **distinct subjects**, which a naive prefix regex
would silently merge. Only the ITA file's own `subject` column is
authoritative. Asserted to yield exactly 185 images / 28 subjects.

In [ ]:
SUBJECT_CSV = """stem,subject
TAM006_V5_UA_N,TAM006
TAM006_V17_UA_N,TAM006
TAM0009_V2_UA_N,TAM0009
TAM009_V4_UA_N,TAM009
TAM028_V13_UA_N,TAM028
TAM030_V10_UA_N,TAM030
TAM030_V11_UA_N,TAM030
TAM030_V12_UA_N,TAM030
TAM030_V13_UA_N,TAM030
TAM031_V1_UA_N,TAM031
TAM031_V2_UA_N,TAM031
TAM031_V4_UA_N,TAM031
TAM033_V10_UA_N,TAM033
TAM033_V11_UA_N,TAM033
TAM033_V13_UA_N,TAM033
TAM033_V15_UA_N,TAM033
TAM034_V4_UA_N,TAM034
TAM034_V5_UA_N,TAM034
TAM034_V9_LA_N,TAM034
TAM034_V11_UA_N,TAM034
TAM034_V14_UA_N,TAM034
TAM034_V15_UA_N,TAM034
TAM034_V17_UA_N,TAM034
TAM038_V1_UA_N,TAM038
TAM038_V2_UA_N,TAM038
TAM038_V3_UA_N,TAM038
TAM038_V5_UA_N,TAM038
TAM038_V11_UA_N,TAM038
TAM038_V12_UA_N,TAM038
TAM047_V12_UA_N,TAM047
TAM047_V15_UA_N,TAM047
TAM047_V16_UA_N,TAM047
TAM047_V19_UA_N,TAM047
TAM047_V20_UA_N,TAM047
TAM057_V9_UA_N,TAM057
TAM057_V10_UA_N,TAM057
TAM057_V11_LA_N,TAM057
TAM057_V11_UA_N,TAM057
TAM057_V12_LA_N,TAM057
TAM057_V13_LA_N,TAM057
TAM057_V13_UA_N,TAM057
TAM057_V14_LA_N,TAM057
TAM057_V14_UA_N,TAM057
TAM057_V15_UA_N,TAM057
TAM058_V1_UA_N,TAM058
TAM058_V2_UA_N,TAM058
TAM058_V3_UA_N,TAM058
TAM058_V4_UA_N,TAM058
TAM058_V5_UA_N,TAM058
TAM058_V6_UA_N,TAM058
TAM058_V8_UA_N,TAM058
TAM058_V9_UA_N,TAM058
TAM058_V10_UA_N,TAM058
TAM058_V12_UA_N,TAM058
TAM058_V13_UA_N,TAM058
TAM058_V14_UA_N,TAM058
TAM058_V15_UA_N,TAM058
TAM058_V19_UA_N,TAM058
TAM063_V1_UA_N,TAM063
TAM063_V2_UA_N,TAM063
TAM063_V3_UA_N,TAM063
TAM063_V4_UA_N,TAM063
TAM063_V5_UA_N,TAM063
TAM063_V6_UA_N,TAM063
TAM063_V7_UA_N,TAM063
TAM063_V8_UA_N,TAM063
TAM063_V10_UA_N,TAM063
TAM063_V12_UA_N,TAM063
TAM063_V13_UA_N,TAM063
TAM063_V15_UA_N,TAM063
TAM063_V16_UA_N,TAM063
TAM063_V17_UA_N,TAM063
TAM063_V18_UA_N,TAM063
TAM063_V19_UA_N,TAM063
TAM063_V20_UA_N,TAM063
TAM063_V21_UA_N,TAM063
TAM066_V20_UA_N,TAM066
TAM067_V1_UA_N,TAM067
TAM067_V2_UA_N,TAM067
TAM067_V3_UA_N,TAM067
TAM067_V4_UA_N,TAM067
TAM067_V5_UA_N,TAM067
TAM067_V6_UA_N,TAM067
TAM067_V7_UA_N,TAM067
TAM067_V8_UA_N,TAM067
TAM067_V9_UA_N,TAM067
TAM067_V10_UA_N,TAM067
TAM067_V11_UA_N,TAM067
TAM069_V9_UA_N,TAM069
TAM076_V3_UA_N,TAM076
TAM076_V4_UA_N,TAM076
TAM076_V5_UA_N,TAM076
TAM076_V6_UA_N,TAM076
TAM076_V7_UA_N,TAM076
TAM076_V8_UA_N,TAM076
TAM076_V9_UA_N,TAM076
TAM076_V10_UA_N,TAM076
TAM076_V11_UA_N,TAM076
TAM076_V12_UA_N,TAM076
TAM077_V17_UA_N,TAM077
TAM077_V18_UA_N,TAM077
TAM077_V19_UA_N,TAM077
TAM077_V20_UA_N,TAM077
TAM077_V21_UA_N,TAM077
TAM079_V4_UA_N,TAM079
TAM079_V5_UA_N,TAM079
TAM079_V6_UA_N,TAM079
TAM079_V7_UA_N,TAM079
TAM079_V8_UA_N,TAM079
TAM079_V9_UA_N,TAM079
TAM079_V10_UA_N,TAM079
TAM079_V11_UA_N,TAM079
TAM079_V12_UA_N,TAM079
TAM080_V2_UA_N,TAM080
TAM080_V3_UA_N,TAM080
TAM080_V4_UA_N,TAM080
TAM080_V5_UA_N,TAM080
TAM080_V6_UA_N,TAM080
TAM080_V7_UA_N,TAM080
TAM080_V8_UA_N,TAM080
TAM080_V10_UA_N,TAM080
TAM080_V14_UA_N,TAM080
TAM080_V17_UA_N,TAM080
TAM081_V10_UA_N,TAM081
TAM081_V11_UA_N,TAM081
TAM081_V12_UA_N,TAM081
TAM082_V1_UA_N,TAM082
TAM082_V2_UA_N,TAM082
TAM082_V3_UA_N,TAM082
TAM082_V5_UA_N,TAM082
TAM082_V6_UA_N,TAM082
TAM083_V1_UA_N,TAM083
TAM083_V2_UA_N,TAM083
TAM083_V3_UA_N,TAM083
TAM083_V4_UA_N,TAM083
TAM083_V5_UA_N,TAM083
TAM083_V8_UA_N,TAM083
TAM083_V9_UA_N,TAM083
TAM084_V2_UA_N,TAM084
TAM084_V3_UA_N,TAM084
TAM084_V5_UA_N,TAM084
TAM084_V6_UA_N,TAM084
TAM084_V7_UA_N,TAM084
TAM084_V9_UA_N,TAM084
TAM084_V10_UA_N,TAM084
TAM085_V1_UA_N,TAM085
TAM085_V3_UA_N,TAM085
TAM085_V4_UA_N,TAM085
TAM085_V12_UA_N,TAM085
TAM085_V13_LA_N,TAM085
TAM085_V13_UA_N,TAM085
TAM085_V15_LA_N,TAM085
TAM085_V15_UA_N,TAM085
TAM085_V16_LA_N,TAM085
TAM085_V16_UA_N,TAM085
TAM085_V17_LA_N,TAM085
TAM085_V17_UA_N,TAM085
TAM085_V18_UA_N,TAM085
TAM085_V19_LA_N,TAM085
TAM085_V19_UA_N,TAM085
TAM085_V21_LA_N,TAM085
TAM086_V3_UA_N,TAM086
TAM086_V4_UA_N,TAM086
TAM086_V5_UA_N,TAM086
TAM086_V6_UA_N,TAM086
TAM086_V7_UA_N,TAM086
TAM086_V8_UA_N,TAM086
TAM086_V9_UA_N,TAM086
TAM086_V10_UA_N,TAM086
TAM088_V1_UA_N,TAM088
TAM088_V2_UA_N,TAM088
TAM088_V3_UA_N,TAM088
TAM088_V5_UA_N,TAM088
TAM088_V11_UA_N,TAM088
TAM088_V12_UA_N,TAM088
TAM088_V13_UA_N,TAM088
TAM088_V15_UA_N,TAM088
TAM088_V16_UA_N,TAM088
TAM088_V18_UA_N,TAM088
TAM088_V19_UA_N,TAM088
TAM088_V20_UA_N,TAM088
TAM100_V10_UA_N,TAM100
TAM100_V15_UA_N,TAM100
TAM100_V16_UA_N,TAM100
TAM100_V17_UA_N,TAM100
"""
subjects = pd.read_csv(io.StringIO(SUBJECT_CSV))
assert len(subjects) == 185, f'expected 185 subject rows, got {len(subjects)}'
assert subjects.subject.nunique() == 28, f'expected 28 subjects, got {subjects.subject.nunique()}'
assert subjects.stem.is_unique, 'duplicate stems in subject map'
# fail loudly if the manifest and the subject map disagree
_missing = set(test_df['stem']) - set(subjects['stem'])
assert not _missing, f'{len(_missing)} test stems have no subject: {sorted(_missing)[:5]}'
print('subject map OK:', len(subjects), 'images /', subjects.subject.nunique(), 'subjects')
print('sanity -- TAM009 and TAM0009 are distinct:',
      sorted(s for s in subjects.subject.unique() if s.startswith('TAM00')))

## 7. Model registry

Fair-KD (A/B) and the ALS->WL models are **not in this package**. They are
listed as `missing` and skipped rather than back-filled with historical
numbers from another run.

In [ ]:
REGISTRY = [
    # run_name, display, family, pretrained_key (SegFormer only)
    ('segformer_b2_teacher',    'SegFormer-B2 (teacher)',   'segformer', 'segformer_b2_pretrained'),
    ('segformer_b0_direct',     'SegFormer-B0 (direct)',    'segformer', 'segformer_b0_pretrained'),
    ('segformer_b0_distilled',  'SegFormer-B0 (distilled)', 'segformer', 'segformer_b0_pretrained'),
    ('yolo_sem_direct',         'YOLO26n-sem (direct)',     'yolo',      None),
    ('yolo_sem_distilled',      'YOLO26n-sem (distilled)',  'yolo',      None),
]
NOT_IN_PACKAGE = [
    'segformer_b0_fairness_distill_approach_a',
    'segformer_b0_fairness_distill_approach_b',
    'segformer_b0_als_to_wl_distilled',
]
for m in NOT_IN_PACKAGE:
    print(f'  missing (not in package, skipped -- NOT substituted): {m}')
print()
for run, disp, fam, _ in REGISTRY:
    p = f'{run}/best_model.pt' if fam == 'segformer' else f'{run}/ultralytics_runs/train/weights/best.pt'
    print(f'  {"OK  " if os.path.exists(p) else "MISS"} {disp:26s} {p}')

## 8. SegFormer evaluation

Preprocessing is `pipeline.data.get_augmentation(training=False, 640, 640)`
(ImageNet normalisation) via `BruiseDataset` -- correct for SegFormer, since
that is exactly how these weights were trained. The operating point is the
**validation-selected threshold returned by `load_segformer_model`**; it is not
discarded and no argmax/logit-sign rule is used. Prediction is
`sigmoid(logits[:,0]) >= threshold`.

Expected: B2 tau=0.75 -> median Dice 0.827 | B0-direct tau=0.60 -> 0.811 |
B0-distilled tau=0.55 -> 0.828.

In [ ]:
EXPECTED_SEGFORMER = {           # (threshold, median_dice) from the locked test set
    'segformer_b2_teacher':   (0.75, 0.827),
    'segformer_b0_direct':    (0.60, 0.811),
    'segformer_b0_distilled': (0.55, 0.828),
}

per_image_frames = {}
model_meta = {}

loader = DataLoader(BruiseDataset(test_df, IMG_H, IMG_W, training=False),
                    batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

for run, disp, fam, pkey in REGISTRY:
    if fam != 'segformer':
        continue
    model, threshold, ckpt = load_segformer_model(run, pkey, paths, DEVICE)
    model = model.to(DEVICE).eval()
    n_params, _ = count_params(model)

    rows = []
    with torch.no_grad():
        for x, y, stems, _, _ in loader:
            x = x.to(DEVICE, non_blocking=True)
            logits = model(x)                                  # [B,1,640,640]
            prob   = torch.sigmoid(logits[:, 0])               # bruise probability
            pred   = (prob >= threshold).to(torch.uint8).cpu().numpy()
            gt     = (y[:, 0].numpy() > 0.5).astype('uint8')
            for i, stem in enumerate(stems):
                rows.append(compute_image_row(pred[i], gt[i], str(stem)))

    df = pd.DataFrame(rows)
    df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
    per_image_frames[run] = df
    model_meta[run] = {'display': disp, 'family': fam, 'params_m': n_params / 1e6,
                       'tau': float(threshold)}

    exp_tau, exp_med = EXPECTED_SEGFORMER[run]
    print(f'{disp:26s} tau={threshold:.2f} (exp {exp_tau:.2f}) '
          f'median_dice={df.dice.median():.3f} (exp {exp_med:.3f}) '
          f'mean={df.dice.mean():.3f} miss={df.complete_miss.mean()*100:.2f}%')
    del model; torch.cuda.empty_cache()

## 9. YOLO evaluation -- Ultralytics native inference, scored at 640

Mirrors `scripts/34_evaluate_yolo_native_at_640.py`. The details that matter:

* The **original image path** goes to `wrapper.predict(...)`. The images are
  4022x6024; pre-resizing them to a 640 square would destroy the aspect ratio.
  Ultralytics must do its own letterbox, matching `model.train(imgsz=640)`.
* The dense class map is read from **`result.semantic_mask.data`**, not
  `result.masks` -- `yolo26n-sem` is a semantic model and `.masks` is an
  instance-segmentation attribute that is `None` here. `retina_masks` is not
  used.
* `.predict()` returns the class map **already upsampled to native
  4022x6024**, so it is resized down to 640 with `INTER_NEAREST`; the GT is
  brought to 640 the same way, so prediction and GT land on an identical grid.
* Masks are squeezed defensively -- `(H,W,1)` is a real failure mode here.

Expected: direct mean 0.735 / median 0.849 / miss 9.73% |
distilled mean 0.741 / median 0.843 / miss 7.03%.

In [ ]:
EXPECTED_YOLO = {                 # (mean_dice, median_dice, miss_pct)
    'yolo_sem_direct':    (0.735, 0.849, 9.73),
    'yolo_sem_distilled': (0.741, 0.843, 7.03),
}

def to_640_nearest(mask):
    """Binary mask -> 640x640, nearest-neighbour. Squeezes a trailing
    singleton axis first: cv2 can hand back (H,W,1), and cv2.resize would
    silently preserve that axis and break the (H,W) metric functions."""
    m = np.asarray(mask)
    if m.ndim == 3:
        m = m.squeeze(-1)
    r = cv2.resize(m.astype('uint8'), (IMG_W, IMG_H), interpolation=cv2.INTER_NEAREST)
    return (r > 0).astype('uint8')

for run, disp, fam, _ in REGISTRY:
    if fam != 'yolo':
        continue
    best_pt = f'{run}/ultralytics_runs/train/weights/best.pt'
    wrapper = YOLO(best_pt)
    n_params = sum(p.numel() for p in wrapper.model.parameters())

    rows = []
    for i, r in enumerate(test_df.itertuples(index=False), start=1):
        # native inference: Ultralytics letterboxes + scales to [0,1] internally
        res = wrapper.predict(source=str(r.image_path), imgsz=IMG_H, device=0, verbose=False)[0]
        cm = res.semantic_mask.data                     # NOT res.masks (None for semantic)
        if hasattr(cm, 'cpu'):
            cm = cm.cpu().numpy()
        pred_native = (np.asarray(cm) == 1).astype('uint8')   # already at 4022x6024
        gt_native   = read_gt_mask(str(r.mask_path))
        rows.append(compute_image_row(to_640_nearest(pred_native),
                                      to_640_nearest(gt_native), str(r.stem)))
        if i % 50 == 0:
            print(f'  [{run}] {i}/{len(test_df)}')

    df = pd.DataFrame(rows)
    df['complete_miss'] = (df.pred_positive_pixels == 0) & (df.gt_positive_pixels > 0)
    per_image_frames[run] = df
    model_meta[run] = {'display': disp, 'family': fam, 'params_m': n_params / 1e6,
                       'tau': None}      # native argmax -- no tunable threshold

    em, emd, ems = EXPECTED_YOLO[run]
    print(f'{disp:26s} mean={df.dice.mean():.3f} (exp {em:.3f}) '
          f'median={df.dice.median():.3f} (exp {emd:.3f}) '
          f'miss={df.complete_miss.mean()*100:.2f}% (exp {ems:.2f}%)')

## 10. Sanity checks

The decisive one: YOLO's blank-mask fraction must be ~0.07-0.10. A value near
1.0 would mean the model is emitting empty masks -- the signature of a broken
inference path (wrong preprocessing, or reading the wrong result attribute).

In [ ]:
ok = True
for run, meta in model_meta.items():
    df = per_image_frames[run]
    blank = float((df.pred_positive_pixels == 0).mean())
    assert len(df) == 185, f'{run}: {len(df)} rows, expected 185'
    if meta['family'] == 'yolo':
        good = 0.05 <= blank <= 0.12
        ok &= good
        print(f'{meta["display"]:26s} blank-mask fraction = {blank:.3f} '
              f'{"OK (expected ~0.07-0.10)" if good else "*** OUT OF RANGE -- inference path suspect ***"}')
    else:
        print(f'{meta["display"]:26s} blank-mask fraction = {blank:.3f}')
print()
print('SANITY:', 'PASS' if ok else '*** FAIL -- do not trust the numbers above ***')

## 11. Speed benchmark (resize NOT timed)

**Timed window:** raw forward pass + postprocess-to-binary-mask, with the
preprocessed tensor **already on the GPU before the timer starts**.

**Excluded from the timed window:** disk read, decode, resize/normalise, and
host-to-device transfer. This is an architecture-speed number, not an
end-to-end camera-to-overlay number, and it upper-bounds deployable frame rate.

`torch.cuda.synchronize()` brackets every timed call (GPU work is async, so
without it we would time kernel *launches*, not compute). 30 warmup iterations,
100 timed repeats, `cudnn.benchmark = True` (safe here: the input shape is
fixed at 640x640, so autotuning cannot thrash).

YOLO is timed as raw module forward + **argmax**, which is exactly Ultralytics'
native decision rule -- no threshold, no temperature. `wrapper.predict()` is not
timed because it bundles file I/O and preprocessing that this benchmark
explicitly excludes.

In [ ]:
import copy, statistics
torch.backends.cudnn.benchmark = True     # fixed 640x640 shape -> autotune is safe

N_WARMUP, N_REPEATS = 30, 100
bench_img = test_df.iloc[0]['image_path']

# Build ONE preprocessed tensor on the GPU, outside every timed window.
_tfm = get_augmentation(training=False, img_h=IMG_H, img_w=IMG_W)
_img = cv2.cvtColor(cv2.imread(str(bench_img)), cv2.COLOR_BGR2RGB)
x_seg = _tfm(image=_img)['image'].float().unsqueeze(0).to(DEVICE)          # ImageNet-norm (SegFormer)
x_yolo = torch.from_numpy(cv2.resize(_img, (IMG_W, IMG_H))).permute(2, 0, 1)\
             .float().div(255.0).unsqueeze(0).to(DEVICE)                    # [0,1] (YOLO)

def timed(fn, device):
    for _ in range(N_WARMUP):
        fn()
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)
    ts = []
    for _ in range(N_REPEATS):
        torch.cuda.synchronize(device)
        t0 = time.perf_counter()
        fn()
        torch.cuda.synchronize(device)
        ts.append((time.perf_counter() - t0) * 1000.0)
    peak = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    return {'mean_ms': statistics.fmean(ts), 'median_ms': statistics.median(ts),
            'fps_from_mean': 1000.0 / statistics.fmean(ts),
            'peak_gpu_mem_mb': peak}

bench = {}
for run, disp, fam, pkey in REGISTRY:
    if fam == 'segformer':
        model, threshold, _ = load_segformer_model(run, pkey, paths, DEVICE)
        model = model.to(DEVICE).eval()
        def fn(m=model, t=threshold):
            with torch.no_grad():
                logits = m(x_seg)
                _ = (torch.sigmoid(logits[:, 0]) >= t).to(torch.uint8)
        bench[run] = timed(fn, DEVICE)
        del model
    else:
        wrapper = YOLO(f'{run}/ultralytics_runs/train/weights/best.pt')
        nn_model = copy.deepcopy(wrapper.model).to(DEVICE).eval()
        def fn(m=nn_model):
            with torch.no_grad():
                p = m(x_yolo)
                if isinstance(p, (tuple, list)):
                    p = p[0]
                if p.shape[-2:] != (IMG_H, IMG_W):
                    p = F.interpolate(p.float(), size=(IMG_H, IMG_W), mode='bilinear', align_corners=False)
                _ = p.argmax(dim=1).to(torch.uint8)      # native decision rule
        bench[run] = timed(fn, DEVICE)
        del nn_model
    torch.cuda.empty_cache()
    b = bench[run]
    print(f'{model_meta[run]["display"]:26s} {b["mean_ms"]:7.3f} ms  {b["fps_from_mean"]:7.1f} FPS  '
          f'peak {b["peak_gpu_mem_mb"]:7.1f} MB')

TIMING_SCOPE = ('TIMED: raw forward + postprocess-to-mask, input tensor already on GPU. '
                'NOT TIMED: disk read, decode, resize/normalise, host-to-device transfer. '
                f'{N_WARMUP} warmup iters, {N_REPEATS} repeats, cudnn.benchmark=True, '
                'cuda.synchronize() before and after each timed call.')
print('\n' + TIMING_SCOPE)

## 12. Master comparison table

In [ ]:
rows = []
for run, disp, fam, _ in REGISTRY:
    df, meta, b = per_image_frames[run], model_meta[run], bench[run]
    rows.append({
        'model': disp,
        'training': 'distilled' if 'distill' in run else ('teacher' if 'teacher' in run else 'direct'),
        'params_M': round(meta['params_m'], 2),
        'tau': 'argmax' if meta['tau'] is None else f'{meta["tau"]:.2f}',
        'median_dice': round(df.dice.median(), 4),
        'mean_dice': round(df.dice.mean(), 4),
        'median_iou': round(df.iou.median(), 4),
        'complete_miss_pct': round(df.complete_miss.mean() * 100, 2),
        'fps': round(b['fps_from_mean'], 1),
        'peak_gpu_mem_mb': round(b['peak_gpu_mem_mb'], 1),
    })
master = pd.DataFrame(rows)
master

## 13. Subject-level (cluster) bootstrap CIs

Resamples **subjects with replacement**, taking all images of each drawn
subject -- *not* images. Images from one subject share skin, lighting, and
injury, so an image-level bootstrap would treat 185 correlated observations as
independent and report intervals that are far too narrow. The effective sample
size here is **28**, not 185.

These are labelled `cluster_bootstrap_*` and are genuinely clustered. Nothing
in this notebook reports an image-level bootstrap under that name.

In [ ]:
RNG = np.random.default_rng(42)
B = 4000

def cluster_boot_ci(df, stat_fn, B=B):
    d = df.merge(subjects, on='stem', how='inner')
    assert len(d) == 185, f'subject merge dropped rows: {len(d)}'
    subs = d.subject.unique()
    by = {s: d[d.subject == s] for s in subs}
    vals = []
    for _ in range(B):
        pick = RNG.choice(subs, size=len(subs), replace=True)
        vals.append(stat_fn(pd.concat([by[s] for s in pick], ignore_index=True)))
    lo, hi = np.percentile(vals, [2.5, 97.5])
    return lo, hi

ci_rows = []
for run, disp, fam, _ in REGISTRY:
    df = per_image_frames[run]
    md_lo, md_hi = cluster_boot_ci(df, lambda d: d.dice.median())
    mn_lo, mn_hi = cluster_boot_ci(df, lambda d: d.dice.mean())
    ms_lo, ms_hi = cluster_boot_ci(df, lambda d: d.complete_miss.mean() * 100)
    ci_rows.append({
        'model': disp,
        'median_dice': round(df.dice.median(), 4),
        'median_dice_CI95': f'[{md_lo:.3f}, {md_hi:.3f}]',
        'mean_dice': round(df.dice.mean(), 4),
        'mean_dice_CI95': f'[{mn_lo:.3f}, {mn_hi:.3f}]',
        'miss_pct': round(df.complete_miss.mean() * 100, 2),
        'miss_pct_CI95': f'[{ms_lo:.2f}, {ms_hi:.2f}]',
    })
cis = pd.DataFrame(ci_rows)
print(f'Subject-level cluster bootstrap, B={B}, resampling 28 subjects (not 185 images).')
cis

## 14. Save results to Drive

In [ ]:
import datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
OUT = f'{LOCAL_DIR}/eval_results'
os.makedirs(OUT, exist_ok=True)

for run in per_image_frames:
    per_image_frames[run].merge(subjects, on='stem', how='left').to_csv(
        f'{OUT}/per_image_{run}.csv', index=False)
master.to_csv(f'{OUT}/master_comparison.csv', index=False)
cis.to_csv(f'{OUT}/cluster_bootstrap_cis.csv', index=False)
json.dump({
    'timing_scope': TIMING_SCOPE,
    'n_images': 185, 'n_subjects': 28,
    'scoring_grid': f'{IMG_H}x{IMG_W}',
    'bootstrap': {'type': 'subject-level cluster bootstrap', 'B': B, 'resampled_unit': 'subject'},
    'gpu': torch.cuda.get_device_name(0),
    'benchmark': {k: bench[k] for k in bench},
    'models_not_in_package': NOT_IN_PACKAGE,
    'excluded': ['ITA/fairness tables (max-min gap CI ~[0.04,0.89] at 28 subjects)',
                 'native-camera-resolution scoring',
                 'YOLO threshold/temperature calibration path'],
}, open(f'{OUT}/metadata.json', 'w'), indent=2)

dest = f'{DRIVE_DIR}/eval_results_{stamp}'
shutil.copytree(OUT, dest)
print('Saved to:', dest)
!ls -la {dest}